# Model 4 — ResNet50 Transfer Learning

**Group**: Group 14  
**Members**: *(update with real names)*  
**Model owner**: P4  
**Architecture**: ResNet50 (ImageNet weights, include_top=False) + GlobalAveragePooling2D + Dense(512) + BatchNorm + Dropout + sigmoid  
**Dataset**: NIH Malaria Cell Images — Parasitized vs Uninfected  
**Date**: June 2026

---
This notebook applies ResNet50 pretrained on ImageNet to malaria cell binary classification.  
ResNet50's residual connections allow training deeper networks without vanishing gradients,  
making it a stronger feature extractor than VGG16 with fewer parameters.

## 1. Environment Setup
Install dependencies, set all random seeds for reproducibility, and verify GPU availability.

In [ ]:
!pip install -q kagglehub tqdm

import os, sys, random
import numpy as np
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', gpus if gpus else 'None — training on CPU')

## 2. Dataset Download
Download the NIH Malaria dataset via `kagglehub`. The path is resolved dynamically — the inner `cell_images` folder is targeted explicitly to avoid the nested directory conflict.

In [ ]:
import kagglehub

path = kagglehub.dataset_download('iarunava/cell-images-for-detecting-malaria')
print('Downloaded to:', path)

# Exclude outer cell_images/ folder — target the directory with ONLY the two class folders
DATA_DIR = None
for root, dirs, _ in os.walk(path):
    if 'Parasitized' in dirs and 'Uninfected' in dirs and 'cell_images' not in dirs:
        DATA_DIR = root
        break

assert DATA_DIR is not None, 'Could not locate Parasitized/Uninfected folders'
print('DATA_DIR:', DATA_DIR)
print('Parasitized images:', len(os.listdir(os.path.join(DATA_DIR, 'Parasitized'))))
print('Uninfected images: ', len(os.listdir(os.path.join(DATA_DIR, 'Uninfected'))))

## 3. Shared Helper Functions
All helper functions are defined inline — this notebook runs independently on any platform (local or Google Colab) without needing `utils.py`.  
**Note**: ResNet50 requires raw pixel values [0, 255] — `resnet50.preprocess_input` is applied inside the model, so `load_dataset` is called with `normalise=False`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve,
)

# ── Constants
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
SEED        = 42
CLASS_NAMES = ['Parasitized', 'Uninfected']
TRAIN_SPLIT = 0.80
VAL_SPLIT   = 0.10
TEST_SPLIT  = 0.10

os.makedirs('figures',     exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# ── Augmentation layer
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='data_augmentation')

# ── Dataset loader (normalise=False for pretrained models)
def load_dataset(data_dir, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, normalise=True):
    full_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        image_size=image_size,
        batch_size=None,
        shuffle=True,
        seed=SEED,
    )
    total   = sum(1 for _ in full_ds)
    n_train = int(total * TRAIN_SPLIT)
    n_val   = int(total * VAL_SPLIT)
    train_ds  = full_ds.take(n_train)
    remaining = full_ds.skip(n_train)
    val_ds    = remaining.take(n_val)
    test_ds   = remaining.skip(n_val)
    AUTOTUNE  = tf.data.AUTOTUNE
    cast_fn = (lambda img, lbl: (tf.cast(img, tf.float32) / 255.0, lbl)) if normalise \
              else (lambda img, lbl: (tf.cast(img, tf.float32), lbl))
    train_ds = (train_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().shuffle(1000, seed=SEED).batch(batch_size).prefetch(AUTOTUNE))
    val_ds   = (val_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().batch(batch_size).prefetch(AUTOTUNE))
    test_ds  = (test_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().batch(batch_size).prefetch(AUTOTUNE))
    return train_ds, val_ds, test_ds

# ── Metrics
def evaluate_model(model, test_ds):
    y_true, y_pred_prob = [], []
    for images, labels in test_ds:
        preds = model(images, training=False).numpy().flatten()
        y_pred_prob.extend(preds)
        y_true.extend(labels.numpy().flatten())
    y_true      = np.array(y_true)
    y_pred_prob = np.array(y_pred_prob)
    y_pred      = (y_pred_prob >= 0.5).astype(int)
    return {
        'accuracy':  round(accuracy_score(y_true, y_pred),     4),
        'precision': round(precision_score(y_true, y_pred),    4),
        'recall':    round(recall_score(y_true, y_pred),       4),
        'f1':        round(f1_score(y_true, y_pred),           4),
        'auc':       round(roc_auc_score(y_true, y_pred_prob), 4),
        'y_true':    y_true,
        'y_pred':    y_pred,
        'y_prob':    y_pred_prob,
    }

# ── Learning curves
def plot_learning_curves(history, experiment_name, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Learning curves — {experiment_name}', fontsize=14, fontweight='bold')
    axes[0].plot(history.history['loss'],     label='Train loss',     linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val loss',       linewidth=2, linestyle='--')
    axes[0].set_title('Loss over epochs'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['accuracy'],     label='Train accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Val accuracy',   linewidth=2, linestyle='--')
    axes[1].set_title('Accuracy over epochs'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    final_train = history.history['accuracy'][-1]
    final_val   = history.history['val_accuracy'][-1]
    gap = final_train - final_val
    if gap > 0.05:
        print(f'Overfitting detected: train {final_train:.3f} vs val {final_val:.3f} (gap={gap:.3f})')
    elif gap < -0.02:
        print(f'Underfitting detected: train {final_train:.3f} vs val {final_val:.3f}')
    else:
        print(f'Good fit: train {final_train:.3f} vs val {final_val:.3f} (gap={gap:.3f})')

# ── Confusion matrix
def plot_confusion_matrix(metrics_dict, class_names, experiment_name, save_path=None):
    cm  = confusion_matrix(metrics_dict['y_true'], metrics_dict['y_pred'])
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax, linewidths=0.5)
    ax.set_title(f'Confusion matrix — {experiment_name}', fontsize=13, fontweight='bold', pad=14)
    ax.set_ylabel('True label', fontsize=11); ax.set_xlabel('Predicted label', fontsize=11)
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, -0.12,
            f'TP={tp}  TN={tn}  FP={fp}  FN={fn}  |  Sensitivity={tp/(tp+fn):.3f}  Specificity={tn/(tn+fp):.3f}',
            ha='center', va='top', transform=ax.transAxes, fontsize=10, color='dimgray')
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# ── ROC curve
def plot_roc_curve(metrics_dict, experiment_name, save_path=None):
    fpr, tpr, _ = roc_curve(metrics_dict['y_true'], metrics_dict['y_prob'])
    auc_val = metrics_dict['auc']
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color='royalblue', lw=2, label=f'ROC curve (AUC = {auc_val:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
    ax.fill_between(fpr, tpr, alpha=0.08, color='royalblue')
    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
    ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
    ax.set_title(f'ROC Curve — {experiment_name}', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# ── Results table
def build_results_table(experiments_list):
    df = pd.DataFrame(experiments_list)
    df = df[['exp_num','description','accuracy','precision','recall','f1','auc','epochs','notes']]
    df.columns = ['Exp #','Description','Accuracy','Precision','Recall','F1','AUC','Epochs','Notes']
    return df.sort_values('F1', ascending=False).reset_index(drop=True)

# ── Callbacks
def get_callbacks(model_name, experiment_num, patience_es=8, patience_lr=4):
    os.makedirs('checkpoints', exist_ok=True)
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=patience_es, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=patience_lr, min_lr=1e-8, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f'checkpoints/{model_name}_exp{experiment_num}.h5',
            monitor='val_accuracy', save_best_only=True, verbose=0),
    ]

# ── Error analysis
def error_analysis(model, test_ds, class_names, n_samples=12):
    misclassified_images, misclassified_labels, misclassified_preds = [], [], []
    for images, labels in test_ds:
        preds        = model(images, training=False).numpy().flatten()
        pred_classes = (preds >= 0.5).astype(int)
        labels_np    = labels.numpy().astype(int).flatten()
        images_np    = images.numpy()
        mask         = pred_classes != labels_np
        misclassified_images.extend(images_np[mask])
        misclassified_labels.extend(labels_np[mask])
        misclassified_preds.extend(preds[mask])
        if len(misclassified_images) >= n_samples:
            break
    n   = min(n_samples, len(misclassified_images))
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle('Error Analysis — Misclassified Samples', fontsize=14, fontweight='bold')
    for i, ax in enumerate(axes.flatten()[:n]):
        img = misclassified_images[i]
        if img.max() > 1.0:
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        ax.imshow(img)
        true_lbl = class_names[int(misclassified_labels[i])]
        pred_lbl = class_names[int(misclassified_preds[i] >= 0.5)]
        conf     = misclassified_preds[i] if pred_lbl == class_names[1] else 1 - misclassified_preds[i]
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({conf:.2f})', fontsize=9, color='red')
        ax.axis('off')
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/P4_error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Helper functions loaded.')

## 4. Load Dataset
ResNet50 requires 224×224 input. `normalise=False` preserves raw [0, 255] pixel values — `resnet50.preprocess_input` inside the model applies ResNet-specific channel normalisation using ImageNet mean/std.

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

train_ds, val_ds, test_ds = load_dataset(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalise=False,  # resnet50.preprocess_input applied inside model
)

n_train = sum(1 for _ in train_ds) * BATCH_SIZE
n_val   = sum(1 for _ in val_ds)   * BATCH_SIZE
n_test  = sum(1 for _ in test_ds)  * BATCH_SIZE
print(f'Train: ~{n_train} | Val: ~{n_val} | Test: ~{n_test}')

## 5. Model Architecture
ResNet50 base (ImageNet weights, top excluded) with a custom head: GlobalAveragePooling2D → Dense(512) → BatchNormalization → Dropout(0.5) → sigmoid.  
BatchNormalization in the head stabilises training when fine-tuning, complementing ResNet50's built-in BN layers.  
`resnet50.preprocess_input` handles channel-wise mean subtraction — never use /255 normalisation with this model.

In [ ]:
from tensorflow.keras.regularizers import l2

def build_resnet50_model(freeze_base=True, fine_tune_layers=None,
                         use_augmentation=False, batch_size=32):
    """
    freeze_base      : freeze entire ResNet50 base
    fine_tune_layers : list of layer name prefixes to unfreeze e.g. ['conv5_block']
    use_augmentation : prepend data_augmentation layer
    batch_size       : stored for reference; dataset must be reloaded externally
    """
    base_model = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = not freeze_base

    if fine_tune_layers is not None:
        base_model.trainable = True
        # Freeze all layers except those whose names start with given prefixes
        for layer in base_model.layers:
            layer.trainable = any(layer.name.startswith(p) for p in fine_tune_layers)
        trainable = [l.name for l in base_model.layers if l.trainable]
        print(f'Fine-tuning {len(trainable)} base layers')

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x      = inputs

    if use_augmentation:
        x = data_augmentation(x)

    x = tf.keras.applications.resnet50.preprocess_input(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs, name='resnet50_transfer'), base_model

# Preview default architecture
m, _ = build_resnet50_model(freeze_base=True)
m.summary()
del m

## 6. Experiment Tracking
All experiment results are appended to `results_log`. The final summary table is displayed after all 7 experiments.

In [ ]:
results_log = []  # Append one dict per experiment — never overwrite

---
## Experiment 1: Fully Frozen Base — Head Only

**Hypothesis**: Training only the Dense head while keeping all ResNet50 layers frozen establishes a transfer learning baseline. ResNet50's deep residual features should map well to malaria cell morphology, achieving high accuracy from scratch in just a few epochs.

**Change made**: `freeze_base=True`, standard head, LR=1e-3, 10 epochs

In [ ]:
EXP_NUM         = 1
EXP_DESCRIPTION = 'Fully frozen base — head only'
EPOCHS          = 10

model1, base1 = build_resnet50_model(freeze_base=True)
model1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history1 = model1.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('resnet50', EXP_NUM), verbose=1,
)

metrics1 = evaluate_model(model1, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics1["accuracy"]}')
print(f'Precision: {metrics1["precision"]}')
print(f'Recall:    {metrics1["recall"]}')
print(f'F1-Score:  {metrics1["f1"]}')
print(f'AUC:       {metrics1["auc"]}')

plot_learning_curves(history1, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P4_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics1['accuracy'], 'precision': metrics1['precision'],
    'recall': metrics1['recall'], 'f1': metrics1['f1'], 'auc': metrics1['auc'],
    'epochs': len(history1.history['loss']), 'notes': '',
})

**Interpretation**: *(How well did frozen ResNet50 features transfer to malaria cells? Compare convergence speed to frozen VGG16.)*

---
## Experiment 2: Fine-tune Last ResNet Block (conv5_block)

**Hypothesis**: Unfreezing the final residual block (conv5_block layers) at a reduced LR=1e-5 allows the highest-level features to adapt to malaria morphology while preserving earlier general-purpose features, reducing the risk of catastrophic forgetting.

**Change made**: Two-stage — Stage 1: frozen head (10 epochs, LR=1e-3) → Stage 2: unfreeze `conv5_block*` layers (10 epochs, LR=1e-5)

In [ ]:
EXP_NUM         = 2
EXP_DESCRIPTION = 'Fine-tune conv5_block (last ResNet block), LR=1e-5'

# Stage 1: head only
model2, base2 = build_resnet50_model(freeze_base=True)
model2.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='binary_crossentropy', metrics=['accuracy'])
print('--- Stage 1: head only (10 epochs) ---')
history2a = model2.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('resnet50', f'{EXP_NUM}a'), verbose=1)

# Stage 2: unfreeze conv5_block layers
base2.trainable = True
for layer in base2.layers:
    layer.trainable = layer.name.startswith('conv5_block')
print(f'Unfrozen: {[l.name for l in base2.layers if l.trainable]}')
model2.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss='binary_crossentropy', metrics=['accuracy'])
print('\n--- Stage 2: fine-tune conv5_block (10 epochs, LR=1e-5) ---')
history2b = model2.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('resnet50', EXP_NUM), verbose=1)

metrics2 = evaluate_model(model2, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics2["accuracy"]}')
print(f'Precision: {metrics2["precision"]}')
print(f'Recall:    {metrics2["recall"]}')
print(f'F1-Score:  {metrics2["f1"]}')
print(f'AUC:       {metrics2["auc"]}')

combined2 = type('H', (), {'history': {
    k: history2a.history[k] + history2b.history[k] for k in history2a.history
}})()
plot_learning_curves(combined2, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P4_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics2['accuracy'], 'precision': metrics2['precision'],
    'recall': metrics2['recall'], 'f1': metrics2['f1'], 'auc': metrics2['auc'],
    'epochs': len(history2a.history['loss']) + len(history2b.history['loss']), 'notes': '',
})

**Interpretation**: *(Did fine-tuning conv5_block improve over frozen base? Any instability in Stage 2 learning curves?)*

---
## Experiment 3: Fine-tune Last 2 ResNet Blocks (conv4 + conv5)

**Hypothesis**: Extending fine-tuning to both conv4_block and conv5_block layers allows mid-to-high level features to adapt to malaria-specific textures. The risk of catastrophic forgetting increases, but the broader feature adaptation may yield better generalisation.

**Change made**: Two-stage — Stage 2 unfreezes both `conv4_block*` and `conv5_block*` layers

In [ ]:
EXP_NUM         = 3
EXP_DESCRIPTION = 'Fine-tune conv4_block + conv5_block (last 2 blocks), LR=1e-5'

# Stage 1
model3, base3 = build_resnet50_model(freeze_base=True)
model3.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='binary_crossentropy', metrics=['accuracy'])
print('--- Stage 1: head only (10 epochs) ---')
history3a = model3.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('resnet50', f'{EXP_NUM}a'), verbose=1)

# Stage 2: unfreeze conv4_block + conv5_block
base3.trainable = True
for layer in base3.layers:
    layer.trainable = layer.name.startswith('conv4_block') or layer.name.startswith('conv5_block')
unfrozen = [l.name for l in base3.layers if l.trainable]
print(f'Unfreezing {len(unfrozen)} layers (conv4+conv5 blocks)')
model3.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss='binary_crossentropy', metrics=['accuracy'])
print('\n--- Stage 2: fine-tune conv4+conv5 blocks (10 epochs, LR=1e-5) ---')
history3b = model3.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('resnet50', EXP_NUM), verbose=1)

metrics3 = evaluate_model(model3, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics3["accuracy"]}')
print(f'Precision: {metrics3["precision"]}')
print(f'Recall:    {metrics3["recall"]}')
print(f'F1-Score:  {metrics3["f1"]}')
print(f'AUC:       {metrics3["auc"]}')

combined3 = type('H', (), {'history': {
    k: history3a.history[k] + history3b.history[k] for k in history3a.history
}})()
plot_learning_curves(combined3, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P4_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics3['accuracy'], 'precision': metrics3['precision'],
    'recall': metrics3['recall'], 'f1': metrics3['f1'], 'auc': metrics3['auc'],
    'epochs': len(history3a.history['loss']) + len(history3b.history['loss']), 'notes': '',
})

**Interpretation**: *(Did 2-block fine-tuning outperform 1-block? Any signs of catastrophic forgetting vs Exp 2?)*

---
## Experiment 4: Smaller Batch Size (batch_size = 16)

**Hypothesis**: Reducing the batch size from 32 to 16 produces noisier but more frequent gradient updates, which can help the model escape sharp minima and improve generalisation. Smaller batches also act as an implicit regulariser.

**Change made**: Dataset reloaded with `batch_size=16`, frozen base, LR=1e-3

In [ ]:
EXP_NUM         = 4
EXP_DESCRIPTION = 'Smaller batch size (batch_size=16), frozen base'
EPOCHS          = 10

# Reload dataset with batch_size=16
train_ds_16, val_ds_16, test_ds_16 = load_dataset(
    data_dir=DATA_DIR, image_size=IMAGE_SIZE, batch_size=16, normalise=False)

model4, base4 = build_resnet50_model(freeze_base=True)
model4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history4 = model4.fit(
    train_ds_16, validation_data=val_ds_16, epochs=EPOCHS,
    callbacks=get_callbacks('resnet50', EXP_NUM), verbose=1,
)

metrics4 = evaluate_model(model4, test_ds_16)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics4["accuracy"]}')
print(f'Precision: {metrics4["precision"]}')
print(f'Recall:    {metrics4["recall"]}')
print(f'F1-Score:  {metrics4["f1"]}')
print(f'AUC:       {metrics4["auc"]}')

plot_learning_curves(history4, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P4_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics4['accuracy'], 'precision': metrics4['precision'],
    'recall': metrics4['recall'], 'f1': metrics4['f1'], 'auc': metrics4['auc'],
    'epochs': len(history4.history['loss']), 'notes': '',
})

**Interpretation**: *(Did batch_size=16 improve generalisation over batch_size=32? Was training more noisy?)*

---
## Experiment 5: Add Data Augmentation

**Hypothesis**: Prepending data augmentation to the ResNet50 pipeline diversifies training inputs, reducing head overfitting while keeping the frozen base features intact. This should improve the train–val gap without degrading the strong ImageNet feature representations.

**Change made**: `use_augmentation=True`, frozen base, batch_size=32

In [ ]:
EXP_NUM         = 5
EXP_DESCRIPTION = 'Data augmentation + frozen base'
EPOCHS          = 10

model5, base5 = build_resnet50_model(freeze_base=True, use_augmentation=True)
model5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history5 = model5.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('resnet50', EXP_NUM), verbose=1,
)

metrics5 = evaluate_model(model5, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics5["accuracy"]}')
print(f'Precision: {metrics5["precision"]}')
print(f'Recall:    {metrics5["recall"]}')
print(f'F1-Score:  {metrics5["f1"]}')
print(f'AUC:       {metrics5["auc"]}')

plot_learning_curves(history5, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P4_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics5['accuracy'], 'precision': metrics5['precision'],
    'recall': metrics5['recall'], 'f1': metrics5['f1'], 'auc': metrics5['auc'],
    'epochs': len(history5.history['loss']), 'notes': '',
})

**Interpretation**: *(Did augmentation improve the train–val gap for frozen ResNet50? Compare with frozen baseline in Exp 1.)*

---
## Experiment 6: SGD + Momentum

**Hypothesis**: SGD with momentum (lr=1e-4, momentum=0.9) may generalise better than Adam for fine-tuning scenarios due to its smoother update trajectory. A lower base LR than Exp 1 compensates for SGD's lack of adaptive learning rates.

**Change made**: `optimizer=SGD(lr=1e-4, momentum=0.9)`, frozen base

In [ ]:
EXP_NUM         = 6
EXP_DESCRIPTION = 'SGD + momentum (lr=1e-4, momentum=0.9), frozen base'
EPOCHS          = 10

model6, base6 = build_resnet50_model(freeze_base=True)
model6.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=1e-4, momentum=0.9),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history6 = model6.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('resnet50', EXP_NUM), verbose=1,
)

metrics6 = evaluate_model(model6, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics6["accuracy"]}')
print(f'Precision: {metrics6["precision"]}')
print(f'Recall:    {metrics6["recall"]}')
print(f'F1-Score:  {metrics6["f1"]}')
print(f'AUC:       {metrics6["auc"]}')

plot_learning_curves(history6, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P4_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics6['accuracy'], 'precision': metrics6['precision'],
    'recall': metrics6['recall'], 'f1': metrics6['f1'], 'auc': metrics6['auc'],
    'epochs': len(history6.history['loss']), 'notes': '',
})

**Interpretation**: *(Did SGD+momentum match Adam for frozen ResNet50 head training? Compare convergence speed and final F1.)*

---
## Experiment 7: Learning Rate Warmup + Cosine Decay

**Hypothesis**: A warmup schedule (linearly increase LR over 500 steps) followed by cosine decay allows the model to stabilise early in training before reaching peak LR, then gradually reducing it. This should reduce the oscillation seen in constant-LR experiments and improve final convergence.

**Change made**: Custom `CosineDecayWithWarmup` schedule replacing Adam with fixed LR

In [ ]:
EXP_NUM         = 7
EXP_DESCRIPTION = 'LR warmup (500 steps) + cosine decay, frozen base'
EPOCHS          = 15

# Cosine decay with linear warmup
steps_per_epoch = len(list(train_ds))
total_steps     = EPOCHS * steps_per_epoch
warmup_steps    = 500

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=total_steps - warmup_steps,
    alpha=1e-6,
)

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, warmup_steps, cosine_schedule):
        self.warmup_steps    = warmup_steps
        self.cosine_schedule = cosine_schedule

    def __call__(self, step):
        warmup_lr = 1e-3 * tf.cast(step, tf.float32) / tf.cast(self.warmup_steps, tf.float32)
        cosine_lr = self.cosine_schedule(tf.maximum(step - self.warmup_steps, 0))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {'warmup_steps': self.warmup_steps}

model7, base7 = build_resnet50_model(freeze_base=True)
model7.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=WarmupCosineDecay(warmup_steps, lr_schedule)),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history7 = model7.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.ModelCheckpoint(
        filepath='checkpoints/resnet50_exp7.h5',
        monitor='val_accuracy', save_best_only=True, verbose=0)],
    verbose=1,
)

metrics7 = evaluate_model(model7, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics7["accuracy"]}')
print(f'Precision: {metrics7["precision"]}')
print(f'Recall:    {metrics7["recall"]}')
print(f'F1-Score:  {metrics7["f1"]}')
print(f'AUC:       {metrics7["auc"]}')

plot_learning_curves(history7, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P4_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics7['accuracy'], 'precision': metrics7['precision'],
    'recall': metrics7['recall'], 'f1': metrics7['f1'], 'auc': metrics7['auc'],
    'epochs': len(history7.history['loss']), 'notes': '',
})

**Interpretation**: *(Did warmup + cosine decay produce smoother curves than constant LR? Did it outperform Adam with fixed LR in Exp 1?)*

---
## 7. Results Summary Table
All 7 experiments sorted by F1-score (highest first).

In [ ]:
import pandas as pd
results_df = build_results_table(results_log)
pd.set_option('display.float_format', '{:.4f}'.format)
display(results_df)

## 8. Best Model — Detailed Evaluation
Identify the experiment with the highest F1-score and run the confusion matrix, ROC curve, and error analysis.

In [ ]:
exp_map = {
    1: (model1, metrics1, test_ds),
    2: (model2, metrics2, test_ds),
    3: (model3, metrics3, test_ds),
    4: (model4, metrics4, test_ds_16),  # Exp 4 used batch_size=16
    5: (model5, metrics5, test_ds),
    6: (model6, metrics6, test_ds),
    7: (model7, metrics7, test_ds),
}

best_exp_num = results_df.iloc[0]['Exp #']
best_model, best_metrics, best_test_ds = exp_map[best_exp_num]
best_description = results_df.iloc[0]['Description']

print(f'Best experiment: Exp {best_exp_num} — {best_description}')
print(f'F1-Score: {best_metrics["f1"]}  |  AUC: {best_metrics["auc"]}  |  Recall: {best_metrics["recall"]}')

### Confusion Matrix
Plots the confusion matrix for the best ResNet50 model with Sensitivity and Specificity annotated for clinical interpretation.

In [ ]:
plot_confusion_matrix(
    best_metrics, CLASS_NAMES,
    f'Best Model (Exp {best_exp_num}): {best_description}',
    save_path='figures/P4_best_confusion_matrix.png',
)

### ROC Curve
Plots the ROC curve and AUC for the best model — key for comparing ResNet50's discriminative ability against VGG16 and the custom CNN baselines.

In [ ]:
plot_roc_curve(
    best_metrics,
    f'Best Model (Exp {best_exp_num}): {best_description}',
    save_path='figures/P4_best_roc_curve.png',
)

### Error Analysis
Displays misclassified test images to identify cell morphologies that ResNet50 residual features struggle to distinguish correctly.

In [ ]:
error_analysis(best_model, best_test_ds, CLASS_NAMES, n_samples=12)

## 9. Model Summary & Report Notes

### Best configuration
- **Experiment**: *(fill in after running)*
- **Architecture**: ResNet50 (ImageNet) + GAP + Dense(512) + BatchNorm + Dropout(0.5) + sigmoid
- **Training strategy**: *(frozen only / two-stage fine-tune)*
- **Key hyperparameters**: *(LR, batch size, epochs, fine-tune layers)*
- **Test metrics**: Accuracy=X, Precision=X, Recall=X, F1=X, AUC=X

### Clinical relevance
*(Discuss Recall/Sensitivity — does ResNet50's residual feature extraction reduce false negatives compared to VGG16 and custom CNNs? What is the clinical implication of the False Negative count?)*

### Observed patterns
- **Frozen vs fine-tuned**: *(Did fine-tuning blocks improve consistently over frozen?)*
- **Fine-tune depth**: *(conv5 only vs conv4+conv5 — which was safer/better?)*
- **Batch size effect**: *(Did batch_size=16 help generalisation?)*
- **Most impactful change**: *(Which experiment produced the largest F1 gain?)*

### Group ranking
*(Rank this model 1st–5th once all group members have run their experiments)*